# 01 · Data and catalog

**Role of this layer.** All market data comes from a small, committed snapshot of
public FRED series (`data/snapshots/*.csv`), described by `data/catalog.json`. The
catalog records a SHA-256 hash of every file. Every load re-hashes the file and
refuses to continue if it changed, so *"which data did this run use?"* is an exact,
checkable fact, and a replay months later can prove it used the same bytes.

The LLM never sees any of the values: only the metadata returned by
`describe_datasets` (id, description, frequency, kind, date range).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.data import describe_datasets, load_catalog, load_dataset, verify_dataset_hash

catalog = load_catalog(settings.catalog_path)
pd.DataFrame(
    [
        {
            "dataset_id": e.dataset_id,
            "kind": e.kind.value,
            "freq": e.frequency.value,
            "start": e.start_date,
            "end": e.end_date,
            "rows": e.n_rows,
            "missing": e.n_missing,
            "sha256": e.sha256[:12],
        }
        for e in catalog.entries.values()
    ]
)

,dataset_id,kind,freq,start,end,rows,missing,sha256
0,SP500,price_level,D,2016-10-17,2026-10-16,2610,96,312d9c822e48
1,NASDAQCOM,price_level,D,1971-02-05,2026-10-15,14529,487,b1c65ade2486
2,VIXCLS,index_level,D,1990-01-02,2026-10-15,9598,302,90b0099e6536
3,DGS10,rate_level,D,1962-01-02,2026-10-15,16903,721,b972c9a2bbfb
4,DGS2,rate_level,D,1976-06-01,2026-10-15,13143,553,48529b32b013
5,DEXSZUS,price_level,D,1971-01-04,2026-10-09,14550,564,b6dd77b9c99e


`kind` drives the guardrails later on: a `rate_level` series must be differenced
(changes in percentage points), never turned into returns (guardrail V07).

Every file still has the hash recorded in the catalog:

In [3]:
{dataset_id: verify_dataset_hash(catalog, dataset_id)[0] for dataset_id in catalog.ids()}

{'DEXSZUS': True,
 'DGS10': True,
 'DGS2': True,
 'NASDAQCOM': True,
 'SP500': True,
 'VIXCLS': True}

Loading a dataset verifies the hash, parses FRED's CSV (both `DATE` and
`observation_date` headers, `.` or empty as missing), drops missing values
(holidays) instead of inventing them, and reports how many were dropped.

In [4]:
loaded = load_dataset(catalog, "NASDAQCOM")
print(f"{loaded.n_rows_raw} rows in the file, {loaded.n_missing} missing values dropped")
loaded.series.tail()

14529 rows in the file, 487 missing values dropped


date
2026-10-09    27366.17
2026-10-12    27245.90
2026-10-13    27204.38
2026-10-14    26692.87
2026-10-15    26713.03
Name: NASDAQCOM, dtype: float64

This is everything the planner LLM is told about the data: metadata, no values.

In [5]:
pd.DataFrame(describe_datasets(catalog, ["NASDAQCOM", "VIXCLS"]))

,dataset_id,title,description,units,frequency,kind,start_date,end_date
0,NASDAQCOM,NASDAQ Composite Index,"NASDAQ Composite equity index, daily close (index level).","Index, Feb 5 1971 = 100",D,price_level,1971-02-05,2026-10-15
1,VIXCLS,CBOE Volatility Index: VIX,"VIX implied volatility index, daily close; a level, not a tradable price.","Index (annualised implied volatility, percent)",D,index_level,1990-01-02,2026-10-15


### What happens if a snapshot is modified?

We copy the data folder to a temporary directory, change one value in the copy,
and try to load it. The original files are not touched.

In [6]:
import shutil
import tempfile
from pathlib import Path

from quantlab.data import DatasetHashMismatchError

tampered_dir = Path(tempfile.mkdtemp()) / "data"
shutil.copytree(settings.data_dir, tampered_dir)
tampered = load_catalog(tampered_dir / "catalog.json")
path = tampered.path_for("DGS10")
path.write_text(path.read_text().replace(",4.06", ",4.07", 1))

try:
    load_dataset(tampered, "DGS10")
except DatasetHashMismatchError as error:
    print("Refused:", error)

Refused: DGS10: file hash 4cc0a72c5c18... does not match catalog hash b972c9a2bbfb...; the snapshot was modified.
